# SIGATI — Sistema de Gestão de Ativos de TI

---

**SIGATI**, um sistema CRUD em Python para gerenciamento de ativos de TI e vulnerabilidades associadas.

## Criando o Repositório do Projeto

Antes de escrever qualquer linha de código, inicializei o repositório Git. Esse hábito de fazer o `git init` logo no começo evita aquela situação de ter o projeto quase pronto e não saber como versionar direito depois.

```bash
git init
git add .
git commit -m "estrutura inicial do projeto"
git checkout -b teste
```

Criei a branch `teste` logo de cara pra poder experimentar sem comprometer a `main`. A intenção era trabalhar nessa branch e só fazer merge quando alguma parte estivesse estável o suficiente.

Depois criei o repositório no GitHub e conectei:

```bash
git remote add origin https://github.com/felipe-conceicao-silva/sigati.git
git push -u origin main
```

## Planejando a Estrutura

Antes de começar a codar de verdade, parei pra pensar em como o projeto seria organizado. Dava pra colocar tudo num arquivo único — funciona pra projetos bem pequenos — mas fica difícil de navegar conforme o sistema cresce e qualquer alteração exige rolar por centenas de linhas pra achar o trecho certo.

A solução foi separar por responsabilidade: cada arquivo cuida de uma coisa específica. Isso tornou mais fácil trabalhar numa parte do sistema sem precisar se preocupar com o resto.

A estrutura final ficou assim:

```text
inventario_seguranca/
├── main.py          → fluxo principal e menu
├── ativos.py        → operações com ativos de TI
├── vulns.py         → operações com vulnerabilidades
├── dados.py         → leitura e gravação dos arquivos JSON
├── ui.py            → entrada, saída e formatação de tela
├── tipos.py         → enumerações (tipos de ativo, severidades, status)
└── data/
    ├── ativos.json
    └── vulns.json
```

PS: Isso mudou algumas vezes durante o desenvolvimento — essa é a versão final.

## Definindo os Tipos — `tipos.py`

Comecei pelo mais simples e que servia de base pra tudo: os enums. O sistema usa vários valores fixos ao longo do código — tipo de ativo, nível de severidade, status de uma vulnerabilidade — e eu queria garantir que esses valores fossem consistentes em todo lugar.

Ao invés de espalhar números soltos pelo código, defini tudo aqui com `Enum` do Python. Fica mais legível e, se precisar adicionar uma nova opção em qualquer das categorias, é só editar esse arquivo.

In [ ]:
# tipos.py — Enumerações de ativos e vulnerabilidades.

from enum import Enum


class TipoAtivo(Enum):
    NOTEBOOK            = 1
    SERVIDOR            = 2
    ROTEADOR            = 3
    SOFTWARE_LICENCIADO = 4
    APLICACAO_WEB       = 5
    BANCO_DE_DADOS      = 6
    IMPRESSORA_REDE     = 7
    ESTACAO_TRABALHO    = 8


class Severidade(Enum):
    BAIXA   = 1
    MEDIA   = 2
    ALTA    = 3
    CRITICA = 4


class Status(Enum):
    ABERTA            = 1
    EM_TRATAMENTO     = 2
    CORRIGIDA         = 3
    ACEITA_COMO_RISCO = 4


## Persistindo os Dados — `dados.py`

Com os tipos definidos, o próximo passo foi resolver a persistência. O sistema precisa salvar as informações entre uma execução e outra.

Cogitei usar SQLite, que já vem embutido no Python, mas pra esse escopo seria um exagero. JSON resolve bem o problema com muito menos código e, de bônus, os arquivos ficam legíveis direto no editor — o que ajudou bastante na hora de debugar.

O módulo ficou responsável por três coisas: criar os arquivos se ainda não existirem, carregar os dados no início do programa e salvar as alterações quando necessário.

Um detalhe que ficou útil: ao carregar os ativos, o módulo já monta um índice `por_hostname`. Assim, buscar um ativo pelo nome é direto, sem precisar percorrer o dicionário inteiro.

In [ ]:
# dados.py — Leitura e gravação dos arquivos JSON.

import json
import os

PASTA      = "data"
ARQ_ATIVOS = os.path.join(PASTA, "ativos.json")
ARQ_VULNS  = os.path.join(PASTA, "vulns.json")


def iniciar():
    """Cria a pasta e os arquivos JSON se ainda não existirem."""
    os.makedirs(PASTA, exist_ok=True)
    for arq in (ARQ_ATIVOS, ARQ_VULNS):
        if not os.path.exists(arq):
            with open(arq, "w", encoding="utf-8") as f:
                json.dump({}, f)


def ler_ativos():
    """Carrega os ativos e monta um índice por hostname para busca rápida."""
    try:
        with open(ARQ_ATIVOS, encoding="utf-8") as f:
            ativos = json.load(f)
    except (json.JSONDecodeError, FileNotFoundError):
        ativos = {}
    por_hostname = {d["hostname"].lower(): k for k, d in ativos.items()}
    return ativos, por_hostname


def salvar_ativos(ativos):
    with open(ARQ_ATIVOS, "w", encoding="utf-8") as f:
        json.dump(ativos, f, ensure_ascii=False, indent=2)


def ler_vulns():
    """Carrega as vulnerabilidades do disco."""
    try:
        with open(ARQ_VULNS, encoding="utf-8") as f:
            return json.load(f)
    except (json.JSONDecodeError, FileNotFoundError):
        return {}


def salvar_vulns(vulns):
    with open(ARQ_VULNS, "w", encoding="utf-8") as f:
        json.dump(vulns, f, ensure_ascii=False, indent=2)


def proximo_id(dic):
    """Retorna o próximo ID disponível com base nas chaves existentes."""
    return max((int(k) for k in dic), default=0) + 1


## Interface com o Usuário — `ui.py`

Logo nas primeiras funções que fui implementar, percebi que ia repetir muito código de leitura e validação de entrada. Toda operação ia precisar pedir um número inteiro, verificar o intervalo, pedir de novo se fosse inválido... Escrever isso em cada lugar seria um pesadelo de manutenção.

Aí resolvi criar o `ui.py` pra centralizar tudo isso. O módulo ficou com três grupos de funções:

- **Tela**: `limpar`, `titulo`, `linha`, `pausar`
- **Leitura com validação**: `ler_inteiro`, `ler_texto`, `escolher`
- **Exibição de dados**: `mostrar_ativo`, `mostrar_vuln`, `nome_enum`

Uma que ficou especialmente útil foi a `buscar_ativo`. Tanto o módulo de ativos quanto o de vulnerabilidades precisam localizar um ativo antes de qualquer operação. Em vez de escrever isso duas vezes, coloquei aqui e importo nos dois.

In [ ]:
# ui.py — Entrada, saída e formatação de tela.

import os
from tipos import TipoAtivo, Severidade, Status

LINHA  = "═" * 62
LINHA2 = "─" * 62


# ─── Tela ─────────────────────────────────────────────────── #

def limpar():
    os.system("cls" if os.name == "nt" else "clear")


def titulo(texto):
    print(f"\n{LINHA}\n  {texto}\n{LINHA}")


def linha(simples=False):
    print(LINHA2 if simples else LINHA)


def pausar():
    input("\n  Pressione Enter para continuar...")


# ─── Leitura com validação ────────────────────────────────── #

def ler_inteiro(prompt, minimo=None, maximo=None):
    """Lê um inteiro com validação de faixa. Repete até entrada válida."""
    while True:
        try:
            entrada = input(prompt).strip()
            if not entrada:
                print("  [!] Campo obrigatório. Digite um número inteiro.")
                continue
            v = int(entrada)
            if minimo is not None and v < minimo:
                print(f"  [!] O valor deve ser maior ou igual a {minimo}.")
                continue
            if maximo is not None and v > maximo:
                print(f"  [!] O valor deve ser menor ou igual a {maximo}.")
                continue
            return v
        except ValueError:
            print("  [!] Entrada inválida. Digite um número inteiro.")
        except EOFError:
            return minimo if minimo is not None else 0


def ler_texto(prompt, obrigatorio=True):
    """Lê um texto com validação de campo vazio."""
    while True:
        try:
            v = input(prompt).strip()
            if obrigatorio and not v:
                print("  [!] Este campo é obrigatório. Tente novamente.")
                continue
            return v
        except EOFError:
            return ""


def escolher(enum_cls, prompt):
    """Exibe as opções de um Enum e lê a escolha do usuário."""
    print("\n  Opções disponíveis:")
    for item in enum_cls:
        print(f"    {item.value}  →  {item.name.replace('_', ' ').title()}")
    return ler_inteiro(f"  {prompt}: ", minimo=1, maximo=len(enum_cls))


# ─── Formatação ───────────────────────────────────────────── #

def nome_enum(enum_cls, valor):
    """Converte o valor inteiro de um Enum em nome legível."""
    try:
        return enum_cls(int(valor)).name.replace("_", " ").title()
    except (ValueError, KeyError):
        return f"Desconhecido ({valor})"


def mostrar_ativo(d):
    linha()
    print(f"  ID            : {d['id']}")
    print(f"  Hostname      : {d['hostname']}")
    print(f"  Responsável   : {d['responsavel']}")
    print(f"  Setor         : {d['setor']}")
    print(f"  Tipo          : {nome_enum(TipoAtivo, d['tipo'])}")
    print(f"  Descrição     : {d.get('descricao') or '—'}")
    print(f"  Vulns assoc.  : {len(d.get('vulnerabilidades', []))}")
    linha()


def mostrar_vuln(v):
    linha(simples=True)
    print(f"  ID          : {v['id']}")
    print(f"  Descrição   : {v['descricao']}")
    print(f"  Categoria   : {v['categoria']}")
    print(f"  Severidade  : {nome_enum(Severidade, v['severidade'])}")
    print(f"  Status      : {nome_enum(Status, v['status'])}")


# ─── Busca de ativo (compartilhada entre ativos.py e vulns.py) ─ #

def buscar_ativo(ativos, por_hostname):
    """Localiza um ativo por ID ou hostname. Retorna (id_str, dados) ou (None, None)."""
    print("\n  Buscar por:")
    print("    1  →  ID do ativo")
    print("    2  →  Hostname")
    op = ler_inteiro("  Opção: ", minimo=1, maximo=2)

    if op == 1:
        id_str = str(ler_inteiro("  ID do ativo: ", minimo=1))
        if id_str in ativos:
            return id_str, ativos[id_str]
    else:
        hostname = ler_texto("  Hostname: ").lower()
        if hostname in por_hostname:
            id_str = por_hostname[hostname]
            return id_str, ativos[id_str]

    print("\n  [!] Ativo não encontrado.")
    return None, None


## Menu Principal — `main.py`

Com toda a base no lugar, escrevi o `main.py`. O objetivo era manter esse arquivo simples de propósito: inicializa os dados, exibe o menu e chama a função certa pra cada opção escolhida.

Toda a lógica fica nos outros módulos — o `main` só orquestra o fluxo.

In [ ]:
# main.py — Sistema de Inventário de Segurança de TI — UFU
# Disciplina: Cibersegurança | Versão 1.0 — 2026/1

import sys
from dados import iniciar, ler_ativos, ler_vulns
from ui import ler_inteiro, limpar
from ativos import cadastrar_ativo, consultar_ativo, atualizar_ativo, remover_ativo, listar_ativos
from vulns import cadastrar_vuln, ver_vulns, atualizar_vuln


def menu():
    print("\n╔══════════════════════════════════════════════════════╗")
    print("║    SISTEMA DE INVENTÁRIO DE SEGURANÇA DE TI — UFU   ║")
    print("╠══════════════════════════════════════════════════════╣")
    print("║  ATIVOS DE TI                                        ║")
    print("║    1  →  Cadastrar ativo                             ║")
    print("║    2  →  Consultar ativo                             ║")
    print("║    3  →  Atualizar ativo                             ║")
    print("║    4  →  Remover ativo                               ║")
    print("║    5  →  Listar todos os ativos                      ║")
    print("╠══════════════════════════════════════════════════════╣")
    print("║  VULNERABILIDADES                                    ║")
    print("║    6  →  Cadastrar vulnerabilidade                   ║")
    print("║    7  →  Visualizar vulnerabilidades de um ativo     ║")
    print("║    8  →  Atualizar status de vulnerabilidade         ║")
    print("╠══════════════════════════════════════════════════════╣")
    print("║    0  →  Sair                                        ║")
    print("╚══════════════════════════════════════════════════════╝")


def main():
    iniciar()
    ativos, por_hostname = ler_ativos()
    vulns = ler_vulns()

    while True:
        limpar()
        menu()
        op = ler_inteiro("\n  Selecione uma opção: ", minimo=0, maximo=8)

        if op == 0:
            print("\n  Encerrando. Até logo!\n")
            sys.exit(0)
        elif op == 1:
            cadastrar_ativo(ativos, por_hostname)
        elif op == 2:
            consultar_ativo(ativos, por_hostname)
        elif op == 3:
            atualizar_ativo(ativos, por_hostname)
        elif op == 4:
            remover_ativo(ativos, por_hostname, vulns)
        elif op == 5:
            listar_ativos(ativos)
        elif op == 6:
            cadastrar_vuln(ativos, por_hostname, vulns)
        elif op == 7:
            ver_vulns(ativos, por_hostname, vulns)
        elif op == 8:
            atualizar_vuln(ativos, por_hostname, vulns)


if __name__ == "__main__":
    main()


## Gerenciando Ativos — `ativos.py`

Aqui está a parte principal do sistema: o CRUD de ativos. Implementei cinco operações:

- **Cadastrar**: coleta os dados, verifica duplicidade de ID e hostname, e salva.
- **Consultar**: localiza o ativo por ID ou hostname e exibe os detalhes.
- **Atualizar**: permite editar campos sem recriar o ativo inteiro — pressionar Enter mantém o valor atual.
- **Remover**: deleta o ativo e todas as vulnerabilidades vinculadas a ele. Deixar registros soltos sem referência causaria problemas na exibição.
- **Listar**: exibe um resumo de todos os ativos em formato de tabela.

In [ ]:
# ativos.py — CRUD de ativos de TI.

from tipos import TipoAtivo
from dados import salvar_ativos, salvar_vulns
from ui import (
    titulo, linha, pausar,
    ler_inteiro, ler_texto, escolher, nome_enum,
    mostrar_ativo, buscar_ativo,
)


def cadastrar_ativo(ativos, por_hostname):
    titulo("Cadastrar Ativo de TI")

    while True:
        novo_id = ler_inteiro("  ID do ativo (inteiro único): ", minimo=1)
        if str(novo_id) in ativos:
            print(f"  [!] ID {novo_id} já está em uso.")
        else:
            break

    while True:
        hostname = ler_texto("  Nome / Hostname: ")
        if hostname.lower() in por_hostname:
            print(f"  [!] Hostname '{hostname}' já cadastrado.")
        else:
            break

    responsavel = ler_texto("  Responsável: ")
    setor       = ler_texto("  Setor / Localização: ")
    tipo        = escolher(TipoAtivo, "Tipo do ativo")
    descricao   = ler_texto("  Descrição (opcional — Enter para pular): ", obrigatorio=False)

    id_str = str(novo_id)
    ativos[id_str] = {
        "id"              : novo_id,
        "hostname"        : hostname,
        "responsavel"     : responsavel,
        "setor"           : setor,
        "tipo"            : tipo,
        "descricao"       : descricao,
        "vulnerabilidades": [],
    }
    por_hostname[hostname.lower()] = id_str

    salvar_ativos(ativos)
    print(f"\n  [✓] Ativo '{hostname}' cadastrado (ID: {novo_id}).")
    pausar()


def consultar_ativo(ativos, por_hostname):
    titulo("Consultar Ativo de TI")
    id_str, dados = buscar_ativo(ativos, por_hostname)
    if dados:
        mostrar_ativo(dados)
    pausar()


def listar_ativos(ativos):
    titulo("Listar Todos os Ativos")
    if not ativos:
        print("  Nenhum ativo cadastrado.")
        pausar()
        return

    print(f"\n  {'ID':<6} {'Hostname':<22} {'Responsável':<20} {'Tipo'}")
    linha(simples=True)
    for id_str in sorted(ativos, key=lambda x: int(x)):
        d = ativos[id_str]
        print(f"  {d['id']:<6} {d['hostname']:<22} {d['responsavel']:<20} {nome_enum(TipoAtivo, d['tipo'])}")
    linha(simples=True)
    print(f"  Total: {len(ativos)} ativo(s)")
    pausar()


def atualizar_ativo(ativos, por_hostname):
    titulo("Atualizar Ativo de TI")
    id_str, dados = buscar_ativo(ativos, por_hostname)
    if not dados:
        pausar()
        return

    print(f"\n  Editando: {dados['hostname']} (ID {dados['id']})")
    print("  [ Enter sem digitar = manter valor atual ]\n")

    responsavel = ler_texto(f"  Responsável [{dados['responsavel']}]: ", obrigatorio=False)
    setor       = ler_texto(f"  Setor [{dados['setor']}]: ", obrigatorio=False)
    descricao   = ler_texto(f"  Descrição [{dados.get('descricao') or '—'}]: ", obrigatorio=False)

    print(f"\n  Tipo atual: {nome_enum(TipoAtivo, dados['tipo'])}")
    if ler_texto("  Alterar tipo? (s/N): ", obrigatorio=False).lower() == "s":
        dados["tipo"] = escolher(TipoAtivo, "Novo tipo")

    if responsavel: dados["responsavel"] = responsavel
    if setor:       dados["setor"]       = setor
    if descricao:   dados["descricao"]   = descricao

    ativos[id_str] = dados
    salvar_ativos(ativos)
    print("\n  [✓] Ativo atualizado.")
    pausar()


def remover_ativo(ativos, por_hostname, vulns):
    titulo("Remover Ativo de TI")
    id_str, dados = buscar_ativo(ativos, por_hostname)
    if not dados:
        pausar()
        return

    qtd = len(dados.get("vulnerabilidades", []))
    print(f"\n  Ativo: {dados['hostname']} (ID {dados['id']})")
    if qtd:
        print(f"  [!] {qtd} vulnerabilidade(s) associada(s) também será(ão) removida(s).")

    if ler_texto("  Confirmar remoção? (s/N): ", obrigatorio=False).lower() != "s":
        print("  Operação cancelada.")
        pausar()
        return

    for vid in dados.get("vulnerabilidades", []):
        vulns.pop(str(vid), None)

    por_hostname.pop(dados["hostname"].lower(), None)
    del ativos[id_str]

    salvar_ativos(ativos)
    salvar_vulns(vulns)
    print("\n  [✓] Ativo e vulnerabilidades removidos.")
    pausar()


## Gerenciando Vulnerabilidades — `vulns.py`

Com os ativos funcionando, implementei o módulo de vulnerabilidades. A lógica segue um padrão parecido, mas com uma restrição: toda vulnerabilidade precisa estar vinculada a um ativo existente.

Ao cadastrar uma vulnerabilidade, o ID dela vai pra dois lugares: no dicionário de vulnerabilidades e na lista `vulnerabilidades` do ativo correspondente. Isso torna eficiente a consulta por ativo, sem precisar percorrer todas as vulns cadastradas.

Pra esse módulo, implementei três operações:

- **Cadastrar**: registra uma nova vulnerabilidade associada a um ativo.
- **Visualizar**: lista todas as vulnerabilidades de um ativo específico.
- **Atualizar**: permite alterar o status e a severidade de uma vulnerabilidade.

In [ ]:
# vulns.py — Cadastro, visualização e atualização de vulnerabilidades.

from tipos import Severidade, Status
from dados import salvar_ativos, salvar_vulns, proximo_id
from ui import (
    titulo, linha, pausar,
    ler_inteiro, ler_texto, escolher, nome_enum,
    mostrar_vuln, buscar_ativo,
)


def cadastrar_vuln(ativos, por_hostname, vulns):
    titulo("Cadastrar Vulnerabilidade")
    id_str, ativo = buscar_ativo(ativos, por_hostname)
    if not ativo:
        pausar()
        return

    print(f"\n  Ativo: {ativo['hostname']} (ID {ativo['id']})")

    descricao  = ler_texto("  Descrição da vulnerabilidade: ")
    categoria  = ler_texto("  Categoria (ex: Autenticação, Configuração, Atualização): ")
    severidade = escolher(Severidade, "Severidade")
    status     = escolher(Status, "Status")

    novo_id = proximo_id(vulns)
    vulns[str(novo_id)] = {
        "id"        : novo_id,
        "ativo_id"  : ativo["id"],
        "descricao" : descricao,
        "categoria" : categoria,
        "severidade": severidade,
        "status"    : status,
    }

    ativo.setdefault("vulnerabilidades", []).append(novo_id)
    ativos[id_str] = ativo

    salvar_vulns(vulns)
    salvar_ativos(ativos)
    print(f"\n  [✓] Vulnerabilidade cadastrada (ID: {novo_id}).")
    pausar()


def ver_vulns(ativos, por_hostname, vulns):
    titulo("Visualizar Vulnerabilidades")
    id_str, ativo = buscar_ativo(ativos, por_hostname)
    if not ativo:
        pausar()
        return

    print(f"\n  Ativo: {ativo['hostname']} (ID {ativo['id']})")
    ids = ativo.get("vulnerabilidades", [])

    if not ids:
        print("  [i] Nenhuma vulnerabilidade registrada.")
    else:
        print(f"  Total: {len(ids)} vulnerabilidade(s)\n")
        for vid in ids:
            v = vulns.get(str(vid))
            if v:
                mostrar_vuln(v)
        linha(simples=True)

    pausar()


def atualizar_vuln(ativos, por_hostname, vulns):
    titulo("Atualizar Vulnerabilidade")
    id_str, ativo = buscar_ativo(ativos, por_hostname)
    if not ativo:
        pausar()
        return

    ids = ativo.get("vulnerabilidades", [])
    if not ids:
        print("\n  [i] Nenhuma vulnerabilidade cadastrada para este ativo.")
        pausar()
        return

    print(f"\n  Vulnerabilidades de '{ativo['hostname']}':")
    linha(simples=True)
    for vid in ids:
        v = vulns.get(str(vid))
        if v:
            sev = nome_enum(Severidade, v["severidade"])
            sts = nome_enum(Status, v["status"])
            print(f"  [{v['id']:>3}]  {v['descricao'][:44]:<45}  {sev:<8}  {sts}")
    linha(simples=True)

    vid_sel = ler_inteiro("  ID da vulnerabilidade: ", minimo=1)
    vid_str = str(vid_sel)

    if vid_str not in vulns or vid_sel not in ids:
        print("\n  [!] Vulnerabilidade não encontrada para este ativo.")
        pausar()
        return

    v = vulns[vid_str]
    print(f"\n  Vulnerabilidade: {v['descricao']}")

    if ler_texto("  Alterar status? (s/N): ", obrigatorio=False).lower() == "s":
        v["status"] = escolher(Status, "Novo status")

    if ler_texto("  Alterar severidade? (s/N): ", obrigatorio=False).lower() == "s":
        v["severidade"] = escolher(Severidade, "Nova severidade")

    vulns[vid_str] = v
    salvar_vulns(vulns)
    print("\n  [✓] Vulnerabilidade atualizada.")
    pausar()


## Executando o Sistema

Com todos os arquivos no lugar, rodar o sistema é direto:

```bash
cd inventario_seguranca
python main.py
```

Na primeira execução, o `dados.py` cria automaticamente a pasta `data/` e os arquivos `ativos.json` e `vulns.json`. A partir daí, tudo que for cadastrado fica salvo entre as execuções.

## Resumo

Resumindo, no final cada arquivo ficou responsável por uma parte específica do sistema. O `main.py` controla o fluxo principal, `ativos.py` e `vulns.py` implementam as funcionalidades, `dados.py` cuida da persistência, `ui.py` concentra a interação com o usuário e `tipos.py` reúne os valores padronizados utilizados pelo programa.

A separação ficou bem clara: pra mudar como os dados são salvos, edita só o `dados.py`; pra ajustar a interface, vai no `ui.py`. Isso facilitou bastante durante o desenvolvimento, especialmente nas partes que precisei reescrever.